#  MedTrack_DV

## Milestone 1 – Module 2

# Data Cleaning & Transformation

### Objectives

- Remove duplicate records
- Handle missing values
- Standardize department names
- Normalize healthcare indicators
- Create Tableau-ready dataset

### Deliverables

- hospital_cleaned.csv
- hospital_cleaning.ipynb

### Step 1 – Import Libraries

In [1]:
import pandas as pd
import numpy as np

### Step 2 – Load Raw Dataset

In [2]:
hospital = pd.read_excel("hospital_raw_data.xlsx")

### Step 3 – Check Dataset

In [3]:
hospital.shape

(15757, 72)

In [5]:
hospital.head()

,SNO,MRD No.,D.O.A,D.O.D,AGE,GENDER,RURAL,TYPE OF ADMISSION-EMERGENCY/OPD,month year,DURATION OF STAY,...,State,Total_Beds,ICU_Beds,Doctor_ID,Doctor_Name,Nurses,Staff_Count,Beds_Available,Beds_Occupied,Equipment
0,1,234735,2017-01-04 00:00:00,2017-03-04 00:00:00,81,M,R,E,2017-04-01,3,...,Delhi,1000,200,DOC015,Dr. Suresh,27,62,80,68,ECG Machine
1,2,234696,2017-01-04 00:00:00,2017-05-04 00:00:00,65,M,R,E,2017-04-01,5,...,Karnataka,450,70,DOC008,Dr. Rahul,26,61,80,69,ECG Machine
2,3,234882,2017-01-04 00:00:00,2017-03-04 00:00:00,53,M,U,E,2017-04-01,3,...,Haryana,1250,250,DOC099,Dr. Anita,27,64,80,68,ECG Machine
3,4,234635,2017-01-04 00:00:00,2017-08-04 00:00:00,67,F,U,E,2017-04-01,8,...,Tamil Nadu,400,60,DOC036,Dr. Arjun,26,60,80,65,ECG Machine
4,5,234486,2017-01-04 00:00:00,4/23/2017,60,F,U,E,2017-04-01,23,...,Karnataka,700,100,DOC075,Dr. Suresh,26,60,100,86,Patient Monitor


In [6]:
hospital.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15757 entries, 0 to 15756
Data columns (total 72 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   SNO                              15757 non-null  int64         
 1   MRD No.                          15757 non-null  object        
 2   D.O.A                            15757 non-null  object        
 3   D.O.D                            15757 non-null  object        
 4   AGE                              15757 non-null  int64         
 5   GENDER                           15757 non-null  object        
 6   RURAL                            15757 non-null  object        
 7   TYPE OF ADMISSION-EMERGENCY/OPD  15757 non-null  object        
 8   month year                       15757 non-null  datetime64[ns]
 9   DURATION OF STAY                 15757 non-null  int64         
 10  duration of intensive unit stay  15757 non-null  int64    

### Step 4 – Remove Duplicate Records

# Remove Duplicate Records

Duplicate patient records can affect KPI calculations and dashboard accuracy.

This step removes any duplicate rows from the integrated dataset.

In [8]:
duplicates = hospital.duplicated().sum()

print("Duplicate Records :", duplicates)

Duplicate Records : 0


In [9]:
hospital = hospital.drop_duplicates()

In [10]:
hospital.duplicated().sum()

np.int64(0)

### Step 5 – Handle Missing Values

## Handle Missing Values

Identify missing values and replace or remove them based on the column type.

In [14]:
hospital.isnull().sum().sum()

np.int64(12120)

### Fill Numeric Columns

In [15]:
numeric_cols = hospital.select_dtypes(include=["number"]).columns

hospital[numeric_cols] = hospital[numeric_cols].fillna(
    hospital[numeric_cols].median()
)

### Fill Text Columns

In [16]:
text_cols = hospital.select_dtypes(include="object").columns

hospital[text_cols] = hospital[text_cols].fillna("Unknown")

In [18]:
hospital.isnull().sum().sum()

np.int64(0)

### Step 6 – Standardize Department Names

In [19]:
hospital["Department"] = (
    hospital["Department"]
    .str.strip()
    .str.title()
)

### Step 7 – Normalize Healthcare Indicators

In [22]:
binary_columns = [
    "CAD","ACS","CKD","AKI",
    "STEMI","HFREF","HFNEF"
]
for col in binary_columns:
    hospital[col] = hospital[col].replace(
        {
            1:"Yes",
            0:"No"
        }
    )

### Step 8 – Convert Dates

In [24]:
hospital["D.O.A"].head(10)

0   2017-01-04
1   2017-01-04
2   2017-01-04
3   2017-01-04
4   2017-01-04
5   2017-01-04
6   2017-01-04
7   2017-01-04
8   2017-01-04
9   2017-01-04
Name: D.O.A, dtype: datetime64[ns]

In [25]:
hospital["D.O.D"].head(10)

0    2017-03-04 00:00:00
1    2017-05-04 00:00:00
2    2017-03-04 00:00:00
3    2017-08-04 00:00:00
4              4/23/2017
5    2017-10-04 00:00:00
6    2017-06-04 00:00:00
7              4/13/2017
8    2017-03-04 00:00:00
9    2017-03-04 00:00:00
Name: D.O.D, dtype: object

In [26]:
invalid_doa = pd.to_datetime(hospital["D.O.A"], errors="coerce")

hospital[invalid_doa.isna()][["D.O.A"]]

,D.O.A


In [27]:
invalid_dod = pd.to_datetime(hospital["D.O.D"], errors="coerce")

hospital[invalid_dod.isna()][["D.O.D"]]

,D.O.D
4692,2-1217


In [28]:
hospital["D.O.A"] = pd.to_datetime(
    hospital["D.O.A"],
    errors="coerce"
)

hospital["D.O.D"] = pd.to_datetime(
    hospital["D.O.D"],
    errors="coerce"
)

In [29]:
hospital[["D.O.A", "D.O.D"]].isna().sum()

D.O.A    0
D.O.D    1
dtype: int64

In [30]:
hospital = hospital.dropna(subset=["D.O.D"])

In [31]:
hospital[["D.O.A", "D.O.D"]].isna().sum()

D.O.A    0
D.O.D    0
dtype: int64

In [32]:
hospital.shape

(15756, 72)

In [33]:
hospital["D.O.A"] = pd.to_datetime(hospital["D.O.A"])

hospital["D.O.D"] = pd.to_datetime(hospital["D.O.D"])

### Step 9 – Check Data Types

In [38]:
hospital.info()

<class 'pandas.core.frame.DataFrame'>
Index: 15756 entries, 0 to 15756
Data columns (total 72 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   SNO                              15756 non-null  int64         
 1   MRD No.                          15756 non-null  object        
 2   D.O.A                            15756 non-null  datetime64[ns]
 3   D.O.D                            15756 non-null  datetime64[ns]
 4   AGE                              15756 non-null  int64         
 5   GENDER                           15756 non-null  object        
 6   RURAL                            15756 non-null  object        
 7   TYPE OF ADMISSION-EMERGENCY/OPD  15756 non-null  object        
 8   month year                       15756 non-null  datetime64[ns]
 9   DURATION OF STAY                 15756 non-null  int64         
 10  duration of intensive unit stay  15756 non-null  int64         

## Save the cleaned dataset

In [39]:
hospital.to_excel("hospital_cleaned.xlsx", index=False)

# Module 2 Completed
### Tasks Completed
Duplicate records removed
Missing values handled
Department names standardized
Healthcare indicators normalized
Tableau-ready dataset created
### Deliverables
#### hospital_cleaned. csv
#### hospital_cleaning. ipynb